# 19. FITS gravitational-wave strain: a reproducible signal-analysis workflow

Analyze the project's supplied strain time series without treating it as a confirmed detector event. This notebook combines Astropy FITS I/O and units, NumPy array operations, pandas summaries, SciPy spectral estimation and peak detection, SymPy dimensional algebra, and Matplotlib figures.

All displayed equations use LaTeX. The input is the local illustrative file `simulations/gw_strain.fits`; its provenance and detector calibration are not encoded in its FITS table, so results are descriptive only.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sympy as sp
from astropy import units as u
from astropy.table import Table
from scipy import signal, stats

candidates = [parent / 'simulations' / 'gw_strain.fits' for parent in (Path.cwd(), *Path.cwd().parents)]
data_path = next((path for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError('Could not locate simulations/gw_strain.fits from the current directory or its parents.')
table = Table.read(data_path, hdu=1)
required = {'time', 'strain', 'frequency'}
if not required.issubset(table.colnames):
    raise ValueError(f'Expected FITS fields {sorted(required)}; found {table.colnames}')
df = table.to_pandas().replace([np.inf, -np.inf], np.nan).dropna(subset=['time', 'strain'])
df = df.sort_values('time', kind='stable').drop_duplicates('time').reset_index(drop=True)
if len(df) < 16:
    raise ValueError('At least 16 distinct finite time samples are required for spectral analysis.')
time_s = df['time'].to_numpy(dtype=float) * u.s
strain = df['strain'].to_numpy(dtype=float)
dt_s = np.diff(time_s.to_value(u.s))
if np.any(dt_s <= 0):
    raise ValueError('Time samples must be strictly increasing.')
dt = np.median(dt_s) * u.s
if not np.allclose(dt_s, dt.to_value(u.s), rtol=1e-3, atol=1e-12):
    raise ValueError('Sampling is not sufficiently uniform for this Welch PSD workflow.')
sample_rate = (1 / dt).to(u.Hz)
print(f'Input: {data_path} | samples: {len(df)} | rate: {sample_rate:.6g}')
print(f'Time span: {(time_s[-1] - time_s[0]).to(u.s):.6g} | finite strain: {np.isfinite(strain).all()}')

In [ ]:
# Notebook metadata
__author__ = "Astro Science Research"
__date__ = "2026-10-28"
__version__ = "1.0.0"
__kernel__ = "python3"
import sys
print(f"Python {sys.version}")
print(f"Notebook: {__file__ if '__file__' in dir() else 'interactive'}")


## Sampling and summary statistics

The evenly sampled sequence sets the observation duration, frequency resolution, and Nyquist limit:

$$ f_s = \frac{1}{\Delta t}, \qquad f_{\mathrm{Nyquist}} = \frac{f_s}{2}, \qquad \Delta f \simeq \frac{1}{T}. $$

The table's strain values are unitless in the file. No physical strain calibration is inferred here.

In [ ]:
duration = (time_s[-1] - time_s[0] + dt).to(u.s)
nyquist = (sample_rate / 2).to(u.Hz)
summary = pd.Series(strain, name='strain').describe(percentiles=[0.05, 0.5, 0.95])
summary.loc['median absolute deviation'] = stats.median_abs_deviation(strain, scale='normal')
display(summary.to_frame())
print(f'Duration: {duration:.6g}; nominal resolution: {(1 / duration).to(u.Hz):.6g}; Nyquist: {nyquist:.6g}')
fig, ax = plt.subplots(figsize=(10, 3.5), constrained_layout=True)
ax.plot(time_s.to_value(u.s), strain, lw=0.7, color='tab:blue')
ax.set(xlabel='Time (s)', ylabel='Strain (dimensionless)', title='Supplied FITS strain series')
ax.grid(alpha=0.25)
plt.show()

## Welch power spectral density and candidate peaks

Welch's method averages modified periodograms of overlapping, windowed segments. The one-sided power spectral density is estimated in strain squared per hertz. Peak selection below is a descriptive ranking rule, not a detection significance test.

In [ ]:
segment_length = min(2048, len(strain))
frequencies, psd = signal.welch(strain, fs=sample_rate.to_value(u.Hz), window='hann', nperseg=segment_length, detrend='constant', scaling='density')
psd_unit = u.dimensionless_unscaled**2 / u.Hz
robust_floor = np.median(psd[1:]) if len(psd) > 1 else np.nan
peak_indices, peak_properties = signal.find_peaks(psd, prominence=robust_floor)
ranked = peak_indices[np.argsort(psd[peak_indices])[::-1]][:10]
peaks = pd.DataFrame({'frequency_hz': frequencies[ranked], 'psd': psd[ranked], 'prominence': peak_properties['prominences'][np.argsort(psd[peak_indices])[::-1][:10]] if len(ranked) else []})
display(peaks)
fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
positive = frequencies > 0
ax.loglog(frequencies[positive], psd[positive], color='tab:purple', lw=1)
if len(ranked):
    ax.scatter(frequencies[ranked], psd[ranked], color='tab:orange', s=20, zorder=3, label='Ranked local peaks')
    ax.legend()
ax.set(xlabel='Frequency (Hz)', ylabel=f'PSD ({psd_unit})', title='Welch one-sided PSD')
ax.grid(which='both', alpha=0.2)
plt.show()

## Symbolic sampling check

SymPy verifies the reciprocal relationship used above before applying it to the measured median cadence. This is algebraic bookkeeping, not a physical model of a source.

In [ ]:
delta_t, frequency_symbol = sp.symbols('\\Delta_t f', positive=True)
sample_rate_symbol = 1 / delta_t
nyquist_symbol = sample_rate_symbol / 2
assert sp.simplify(nyquist_symbol - 1 / (2 * delta_t)) == 0
display(sp.Eq(sp.Symbol('f_N'), nyquist_symbol, evaluate=False))
measured_nyquist = (sample_rate / 2).to(u.Hz)
assert np.isclose(measured_nyquist.to_value(u.Hz), nyquist.to_value(u.Hz))
print(f'Symbolic identity verified; measured Nyquist frequency: {measured_nyquist:.6g}')

## Interpretation and limitations

- The FITS table provides time, strain, and a frequency column, but no detector, event, calibration, or uncertainty metadata.
- The plotted peaks are features of this supplied series only; they are not evidence of an astrophysical signal.
- A detection analysis requires calibrated detector data, a noise model, data-quality checks, and a stated statistical hypothesis test.
- Re-run from a clean kernel. The cells are deterministic and create no files.